# BirdCLEF+ 2025 - Pretrained models: A1-to-A2 distillation on soundscape windows

**T4 GPU, internet off, no submission.** The teacher is the original, unadapted A1 model (`sc4000-birdclef-m3-a1-v2s-train`), run on the windows listed in `soundscape_windows.csv` with fast-infer's decode, mel and sigmoid inference. The model is in full eval mode, float32, no AMP. For the soft labels we take the sigmoid probability as `p ** 1.3` and then set values below 0.01 to 0.

The student uses the A2 Perch MLP head recipe (hidden 512, dropout 0.3, max_lr 1e-3, wd 1e-2, 30 epochs, batch 1024, one-cycle pct_start 0.1, seed 42, default bias). It trains on the train-fold train-clip Perch windows with real labels and per-clip weights, plus all soundscape Perch windows, untransformed, with teacher labels and a loss weight of 0.5 per window. Both kinds of window share the same shuffled batches and the same weighted-mean loss. The standardiser comes from the train-fold source windows only. Validation on the val fold follows the original protocol and is only recorded.

**Output:** `teacher_a1_probs.npy` (float32, csv row order), `perch_head_a1distill.npz`, `species.json`, `head_cfg.json`, `a1distill_results.json`.

Set `M3_SMOKE_TEST=1` for a local run on synthetic data.

In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[v] = "1"                # set before numpy loads; each worker uses one core
import time
T_NOTEBOOK = time.time()

import torch
SMOKE = os.environ.get("M3_SMOKE_TEST") == "1"          # local tests only, never set on Kaggle
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("torch.cuda.get_device_name(0):", torch.cuda.get_device_name(0))
elif SMOKE:
    print("M3_SMOKE_TEST=1: no GPU, running the local CPU test")
else:
    raise RuntimeError("No GPU found. Enable the T4 accelerator; this notebook never falls back to CPU.")

In [ ]:
import ast, glob, importlib, json, multiprocessing as mp, sys, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import timm
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
print("DEVICE:", DEVICE, "|", DEVICE_NAME)
torch.set_num_threads(4)
OUT = Path(os.environ.get("M3_OUT_DIR", "/kaggle/working"))

def gpu_sync():
    # CUDA calls return before the work finishes; wait for it so that timings are real.
    if DEVICE == "cuda":
        torch.cuda.synchronize()

CFG = dict(val_frac=0.2, seed=42, secondary_weight=0.5, batch_size=1024, weight_decay=1e-2, epochs=2 if SMOKE else 30)
PERCH = dict(dim=1536, emb="perch_embeddings.npy", windows="perch_windows.csv", target="perch_soundscape_embeddings.npy",
             hidden=512, dropout=0.3, max_lr=1e-3)
SOUNDSCAPE_WEIGHT = 0.5                                           # literal loss weight per soundscape window
TEACHER_POWER, TEACHER_FLOOR = 1.3, 0.01
N_SOUNDSCAPE_WINDOWS = 12000
NUM_WORKERS = 4
TEACHER_BATCH = int(os.environ.get("M3_BATCH_SIZE", "64"))
GUARD_FILES, GUARD_BATCHES = 40, 8
BUDGET_S = float(os.environ.get("M3_BUDGET_S", 30 * 60))          # env override: local guard test only
SAFETY = 1.25
TEACHER_KERNEL = "sc4000-birdclef-m3-a1-v2s-train"                # the original A1, never the a1-adabn kernel

def check_budget(projected, stage, limit=BUDGET_S):
    print(f"TIMING guard ({stage}) | projected {projected / 60:.1f} min (limit {limit / 60:.0f})", flush=True)
    if projected > limit:
        raise RuntimeError(f"Projected {projected / 60:.1f} min exceeds the {limit / 60:.0f} min limit at stage '{stage}', stopping")

def to_py(o):
    if isinstance(o, dict):
        return {k: to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_py(v) for v in o]
    if isinstance(o, np.generic):
        return o.item()
    return o

def find_dir(marker, env):
    if os.environ.get(env):
        return Path(os.environ[env])
    hits = glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
    if len(hits) != 1:
        raise FileNotFoundError(f"expected exactly one {marker} under /kaggle/input, found {hits}")
    return Path(hits[0]).parent

def find_model_dir(kernel):
    if os.environ.get("M3_MODEL_DIR"):      # local tests only
        return Path(os.environ["M3_MODEL_DIR"])
    hits = [h for h in glob.glob("/kaggle/input/**/model.pth", recursive=True) if kernel in h]
    if len(hits) != 1:
        raise RuntimeError(f"expected one model.pth under kernel {kernel}, found {hits}")
    return Path(hits[0]).parent

DATA = find_dir("birdclef-2025/sample_submission.csv", "M3_DATA_DIR")
MELS = find_dir("train_index.csv", "M3_MELS_DIR")
SOUND = find_dir("soundscape_embed_meta.json", "M3_SOUNDSCAPE_DIR")
PERCH_DIR = find_dir("perch_embeddings.npy", "M3_PERCH_DIR")
MODEL_DIR = find_model_dir(TEACHER_KERNEL)
print("Data:", DATA, "| M1 index:", MELS, "| soundscapes:", SOUND, "| perch:", PERCH_DIR, "| teacher:", MODEL_DIR, "| CPUs:", os.cpu_count(), "| smoke:", SMOKE)

A1_CFG = json.load(open(MODEL_DIR / "cfg.json"))
a1_species = json.load(open(MODEL_DIR / "species.json"))
a2_species = json.load(open(PERCH_DIR / "species.json"))
print("A1 cfg norm:", A1_CFG.get("norm"), "| model:", A1_CFG["model"], "| best_val_auc in cfg:", A1_CFG.get("best_val_auc"))
assert A1_CFG.get("norm") == "zscore_minmax", f"expected zscore_minmax, cfg.json has {A1_CFG.get('norm')}"
if not A1_CFG["model"].startswith("tf_efficientnetv2_s"):
    raise RuntimeError(f"expected tf_efficientnetv2_s, cfg.json has {A1_CFG['model']}")
assert a1_species == a2_species, "A1 and A2 species lists differ (names or order)"

## Split and labels (same table, call and seed as the original notebooks)

In [ ]:
index = pd.read_csv(MELS / "train_index.csv")      # same table, in the same order, that M1 split
species = list(pd.read_csv(DATA / "sample_submission.csv", nrows=0).columns[1:])
assert a1_species == species and a2_species == species, "A1/A2 species list differs from sample_submission"
sp2idx = {s: i for i, s in enumerate(species)}
y_idx = index["primary_label"].astype(str).map(sp2idx)
assert y_idx.notna().all(), "primary_label not in submission columns"
y_idx = y_idx.astype(int).values

def parse_secondary(s):
    try:
        labs = ast.literal_eval(s) if isinstance(s, str) else []
    except (ValueError, SyntaxError):
        labs = []
    return [sp2idx[l] for l in labs if l in sp2idx]

sec_idx = [parse_secondary(s) for s in index["secondary_labels"]]

idx_tr, idx_va = train_test_split(
    np.arange(len(index)), test_size=CFG["val_frac"], stratify=y_idx, random_state=CFG["seed"])
assert not set(idx_tr) & set(idx_va)
print(len(index), "clips,", len(species), "species | train", len(idx_tr), "| val", len(idx_va), flush=True)

def macro_auc(y_true, y_pred):
    keep = y_true.sum(axis=0) > 0          # like the competition metric
    return roc_auc_score(y_true[:, keep], y_pred[:, keep], average="macro"), int(keep.sum())

def head_predict(head, emb):
    """Sigmoid probabilities from saved head arrays. Keep identical to the inference notebooks."""
    x = (emb - head["mean"]) / head["std"]
    n_layers = int(head["n_layers"])
    for i in range(n_layers):
        x = x @ head[f"W{i}"].T + head[f"b{i}"]
        if i < n_layers - 1:
            x = np.maximum(x, 0.0)       # ReLU
    return 1.0 / (1.0 + np.exp(-x))

## Source windows and soundscape checks (all before any GPU work)

In [ ]:
class Data:
    pass

def prepare(cfg):
    t0 = time.time(); d = Data()
    val_clips = pd.read_csv(PERCH_DIR / "val_clips.csv")
    if not np.array_equal(val_clips["clip"].values, idx_va):
        raise RuntimeError("perch: val_clips.csv does not match the recomputed split")
    if "filename" in index.columns and not (index["filename"].astype(str).values[idx_va] == val_clips["filename"].astype(str).values).all():
        raise RuntimeError("perch: val_clips.csv filenames do not match train_index.csv")
    E = np.load(PERCH_DIR / cfg["emb"]).astype(np.float32)
    windows = pd.read_csv(PERCH_DIR / cfg["windows"])
    if E.shape[1] != cfg["dim"] or len(E) != len(windows) or not np.isfinite(E).all():
        raise RuntimeError(f"perch: bad source embeddings {E.shape} vs {len(windows)} windows")
    d.E = E
    clip_of, win_of = windows["clip"].values, windows["win"].values
    d.in_train = np.isin(clip_of, idx_tr); in_val = np.isin(clip_of, idx_va)
    assert not (d.in_train & in_val).any() and (d.in_train | in_val).all()
    Y = np.zeros((len(windows), len(species)), np.float32)
    for row, c in enumerate(clip_of):
        Y[row, sec_idx[c]] = CFG["secondary_weight"]
        Y[row, y_idx[c]] = 1.0
    clip_windows = pd.Series(clip_of).map(pd.Series(clip_of).value_counts()).values
    d.W_t = torch.from_numpy((1.0 / clip_windows).astype(np.float32)); d.Y_t = torch.from_numpy(Y)
    d.rows_va = np.where(in_val & (win_of == 0))[0]
    d.Y_va = np.zeros((len(d.rows_va), len(species)), np.float32)
    d.Y_va[np.arange(len(d.rows_va)), y_idx[clip_of[d.rows_va]]] = 1.0
    d.rows_all_va = np.where(in_val)[0]
    d.va_groups = clip_of[d.rows_all_va]
    d.order_clips = np.unique(d.va_groups)
    d.Y_clip = np.zeros((len(d.order_clips), len(species)), np.float32)
    d.Y_clip[np.arange(len(d.order_clips)), y_idx[d.order_clips]] = 1.0
    print(f"[perch] windows: source {len(E)} | train {d.in_train.sum()} | val {in_val.sum()} | val first-5s {len(d.rows_va)}", flush=True)
    d.prep_s = time.time() - t0
    return d

d = prepare(PERCH)

# Checks on the soundscape side, before any GPU work
windows_s = pd.read_csv(SOUND / "soundscape_windows.csv")
N_S = len(windows_s)
T32 = np.load(SOUND / PERCH["target"]).astype(np.float32)     # untransformed
assert T32.shape[0] == N_S, f"Perch soundscape rows {T32.shape[0]} != {N_S} windows"
assert T32.shape[1] == PERCH["dim"], T32.shape
assert np.isfinite(T32).all(), "non-finite Perch soundscape embeddings"
assert json.load(open(SOUND / "soundscape_embed_meta.json"))["windows"] == N_S
if not SMOKE:
    assert N_S == N_SOUNDSCAPE_WINDOWS, f"expected {N_SOUNDSCAPE_WINDOWS} soundscape windows, got {N_S}"
csv_names = windows_s["filename"].astype(str).values
csv_win = windows_s["win"].astype(int).values
assert not windows_s.duplicated(["filename", "win"]).any(), "duplicate (filename, win) rows in soundscape_windows.csv"
file_names = list(dict.fromkeys(csv_names.tolist()))
paths = [DATA / "train_soundscapes" / Path(n).name for n in file_names]
missing = [str(p) for p in paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"{len(missing)} of {len(paths)} soundscapes missing, e.g. {missing[:3]}")
print(f"{N_S} soundscape windows in {len(file_names)} files", flush=True)

## Worker code
We copied `to_input`, `decode` and `windows_to_inputs` verbatim from the fast-infer notebook.

In [ ]:
LIB_SRC = r"""
import numpy as np
import soundfile as sf
import librosa

CFG, WIN, TOP_DB = None, None, None

def configure(cfg):
    global CFG, WIN, TOP_DB
    CFG, WIN, TOP_DB = cfg, cfg["sr"] * cfg["duration"], float(cfg["norm_top_db"])

def to_input(db):
    # Same as to_input in the A1 notebooks. For norm "v2" this equals M1's (db + 80) / 80.
    db = np.maximum(db - db.max(), -TOP_DB)
    if CFG["norm"] == "zscore_minmax":
        x = (db - db.mean()) / (db.std() + 1e-6)
        return ((x - x.min()) / (x.max() - x.min() + 1e-6)).astype(np.float32)
    return ((db + TOP_DB) / TOP_DB).astype(np.float32)

def decode(path):
    y, _ = sf.read(str(path), dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    n_win = max(1, int(np.ceil(len(y) / WIN)))
    y = np.pad(y, (0, n_win * WIN - len(y)))
    return y.reshape(n_win, WIN)

def windows_to_inputs(batch):
    # One melspectrogram call for the file. Each row gets its own STFT, as in the per-window loop.
    power = librosa.feature.melspectrogram(
        y=batch, sr=CFG["sr"], n_fft=CFG["n_fft"], hop_length=CFG["hop_length"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=CFG["fmax"])
    # ref=np.max and the normalisation stay per window.
    return np.stack([to_input(librosa.power_to_db(m, ref=np.max, top_db=TOP_DB)) for m in power])

def file_windows(path):
    return windows_to_inputs(decode(path))
"""
LIB_DIR = Path(tempfile.mkdtemp(prefix="a1distill_lib_"))     # outside the output folder
(LIB_DIR / "a1distill_lib.py").write_text(LIB_SRC)
sys.path.insert(0, str(LIB_DIR))
importlib.invalidate_caches()
import a1distill_lib as lib
lib.configure(A1_CFG)

## Teacher labels (original A1 on the soundscape windows)

In [ ]:
t_teacher = time.time()

# 1. Windows from the live audio, in the 4-process CPU pool (spawn: the parent has already touched CUDA)
t0 = time.time()
per_file = []
guard_at = min(GUARD_FILES, len(paths))
with mp.get_context("spawn").Pool(NUM_WORKERS, initializer=lib.configure, initargs=(A1_CFG,)) as pool:
    for i, w in enumerate(pool.imap(lib.file_windows, paths, chunksize=1), 1):
        per_file.append(w)
        if i == guard_at:
            rate = (time.time() - t0) / i
            check_budget(time.time() - T_NOTEBOOK + rate * (len(paths) - i), f"windows, after {i} files")
n_per_file = np.array([len(w) for w in per_file])
WINDOWS = np.concatenate(per_file)
del per_file
WINDOWS_S = time.time() - t0
print(f"windows {WINDOWS.shape} {WINDOWS.dtype} {WINDOWS.nbytes / 1e9:.2f} GB | {WINDOWS_S:.1f}s", flush=True)
assert WINDOWS.dtype == np.float32 and WINDOWS.ndim == 3, (WINDOWS.dtype, WINDOWS.shape)

# 2. Row map from decode order to soundscape_windows.csv order; every (filename, win) must match exactly
row_file = np.repeat(np.array(file_names), n_per_file)
row_win = np.concatenate([np.arange(n) for n in n_per_file])
assert len(WINDOWS) == N_S, f"decoded {len(WINDOWS)} windows, csv has {N_S}"
offset = dict(zip(file_names, np.concatenate([[0], np.cumsum(n_per_file)[:-1]]).tolist()))
pos = np.array([offset[n] + w for n, w in zip(csv_names.tolist(), csv_win.tolist())])
assert pos.min() >= 0 and pos.max() < N_S and len(np.unique(pos)) == N_S, "csv rows do not map one-to-one onto the decoded windows"
assert (row_file[pos] == csv_names).all() and (row_win[pos] == csv_win).all(), "decoded (filename, win) differ from soundscape_windows.csv"

# 3. The teacher: original A1 weights, full eval mode, float32, no AMP, the fast-infer sigmoid
torch.set_num_threads(4)
teacher = timm.create_model(A1_CFG["model"], pretrained=False, in_chans=1, num_classes=len(species))
teacher.load_state_dict(torch.load(MODEL_DIR / "model.pth", map_location="cpu"))
teacher = teacher.to(DEVICE).eval()
assert not any(m.training for m in teacher.modules()), "teacher is not in full eval mode"

t0 = time.time()
n_batches = int(np.ceil(N_S / TEACHER_BATCH))
out = []
with torch.inference_mode():
    for b in range(n_batches):
        xb = torch.from_numpy(WINDOWS[b * TEACHER_BATCH:(b + 1) * TEACHER_BATCH])[:, None].to(DEVICE)
        assert xb.dtype == torch.float32
        out.append(torch.sigmoid(teacher(xb).float()).cpu().numpy())
        if b + 1 == min(GUARD_BATCHES, n_batches):
            gpu_sync()
            check_budget(time.time() - T_NOTEBOOK + (time.time() - t0) / (b + 1) * (n_batches - b - 1), f"teacher, after {b + 1} batches")
gpu_sync()
TEACHER_MODEL_S = time.time() - t0
assert not any(m.training for m in teacher.modules()), "teacher left eval mode"
probs_decode_order = np.concatenate(out).astype(np.float32)
assert probs_decode_order.shape == (N_S, len(species)), probs_decode_order.shape
teacher_p = probs_decode_order[pos]                         # csv row order
assert teacher_p.dtype == np.float32 and np.isfinite(teacher_p).all() and teacher_p.min() >= 0 and teacher_p.max() <= 1
np.save(OUT / "teacher_a1_probs.npy", teacher_p)
del teacher, WINDOWS, out, probs_decode_order
if DEVICE == "cuda":
    torch.cuda.empty_cache()

# 4. Soft labels and statistics (as in a3_distill)
soft = teacher_p ** TEACHER_POWER
soft[soft < TEACHER_FLOOR] = 0
assert soft.dtype == np.float32 and np.isfinite(soft).all() and soft.shape == (N_S, len(species))
n_nonzero = (soft > 0).sum(axis=0); n_half = (soft >= 0.5).sum(axis=0); soft_sum = soft.sum(axis=0)
order = np.lexsort((-soft_sum, -n_nonzero))[:10]        # most windows with a non-zero label, ties by summed soft label
teacher_stats = dict(
    n_windows=N_S, power=TEACHER_POWER, floor=TEACHER_FLOOR, soft_label_stats_are_after_power_and_floor=True,
    fraction_nonzero=float((soft > 0).mean()), mean_max_per_window=float(soft.max(axis=1).mean()),
    windows_any_label_ge_0_5=int((soft.max(axis=1) >= 0.5).sum()),
    top10_species=[dict(species=species[i], windows_nonzero=int(n_nonzero[i]), windows_ge_0_5=int(n_half[i]), sum_soft=float(soft_sum[i])) for i in order])
print(f"Teacher: {N_S} windows | fraction non-zero {teacher_stats['fraction_nonzero']:.4f} | mean max per window {teacher_stats['mean_max_per_window']:.4f} | "
      f"windows with any label >= 0.5: {teacher_stats['windows_any_label_ge_0_5']}", flush=True)
for r in teacher_stats["top10_species"]:
    print("  ", r, flush=True)
TEACHER_S = time.time() - t_teacher

## Student training on the GPU

In [ ]:
def make_head(cfg):
    return nn.Sequential(nn.Linear(cfg["dim"], cfg["hidden"]), nn.ReLU(), nn.Dropout(cfg["dropout"]), nn.Linear(cfg["hidden"], len(species)))

def train_run(cfg, d, X, T32, soft):
    t_start = time.time()
    torch.manual_seed(CFG["seed"]); np.random.seed(CFG["seed"])
    mean, std = X[d.in_train].mean(axis=0), X[d.in_train].std(axis=0) + 1e-6
    # The standardiser comes from the train-fold source windows only; the soundscape windows use it too.
    n_src, n_t = len(X), len(T32)
    Z = torch.from_numpy(np.concatenate([(X - mean) / std, (T32 - mean) / std])).to(DEVICE)
    Y_all = torch.cat([d.Y_t, torch.from_numpy(soft)]).to(DEVICE)
    W_all = torch.cat([d.W_t, torch.full((n_t,), SOUNDSCAPE_WEIGHT, dtype=torch.float32)]).to(DEVICE)
    tr_rows = torch.from_numpy(np.concatenate([np.where(d.in_train)[0], np.arange(n_src, n_src + n_t)]))
    w_src = float(d.W_t.numpy()[d.in_train].sum()); w_sound = SOUNDSCAPE_WEIGHT * n_t
    share = w_sound / (w_src + w_sound)
    print(f"[perch/a1distill] rows: {len(tr_rows)} (source {int(d.in_train.sum())} + soundscape {n_t}) | loss weight: source {w_src:.1f}, soundscape {w_sound:.1f}, soundscape share {share:.4f}", flush=True)

    @torch.no_grad()
    def predict(model, rows):
        model.eval(); out = []
        for i in range(0, len(rows), 4096):
            out.append(torch.sigmoid(model(Z[torch.from_numpy(rows[i:i + 4096]).to(DEVICE)])).cpu().numpy())
        return np.concatenate(out)

    def clip_max_auc(model):
        p = pd.DataFrame(predict(model, d.rows_all_va)).groupby(d.va_groups).max().loc[d.order_clips].values
        return macro_auc(d.Y_clip, p)[0]

    model = make_head(cfg).to(DEVICE)
    epochs = CFG["epochs"]
    steps = int(np.ceil(len(tr_rows) / CFG["batch_size"]))
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["max_lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg["max_lr"], total_steps=epochs * steps, pct_start=0.1)
    best = dict(first=-1.0); name = "perch/a1distill"
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); total = 0.0
        perm = tr_rows[torch.randperm(len(tr_rows))]
        for s in range(steps):
            b = perm[s * CFG["batch_size"]:(s + 1) * CFG["batch_size"]].to(DEVICE)
            bce = nn.functional.binary_cross_entropy_with_logits(model(Z[b]), Y_all[b], reduction="none")
            loss = (bce.mean(1) * W_all[b]).sum() / W_all[b].sum()
            opt.zero_grad(set_to_none=True)
            loss.backward(); opt.step(); sched.step()
            total += loss.item()
        first, n_cls = macro_auc(d.Y_va, predict(model, d.rows_va))
        flag = ""
        if first > best["first"]:
            best = dict(first=first, clip_max=clip_max_auc(model), best_epoch=ep, n_classes=n_cls,
                        state={k: v.clone() for k, v in model.state_dict().items()}); flag = "  <- best"
        print(f"[{name}] ep {ep:2d}/{epochs} | loss {total / steps:.4f} | first-5s {first:.4f} | {time.time() - t0:.1f}s{flag}", flush=True)
        if ep == 1:
            gpu_sync()
            check_budget(time.time() - T_NOTEBOOK + SAFETY * (epochs - 1) * (time.time() - t0), "student, after epoch 1")
    model.load_state_dict(best["state"]); model.eval()

    # Save in the format of the original heads and check the numpy path used at inference
    linears = [m for m in model if isinstance(m, nn.Linear)]
    arrays = dict(mean=mean, std=std, n_layers=len(linears))
    for i, m in enumerate(linears):
        arrays[f"W{i}"] = m.weight.detach().cpu().numpy(); arrays[f"b{i}"] = m.bias.detach().cpu().numpy()
    path = OUT / "perch_head_a1distill.npz"
    np.savez(path, **arrays)
    head = dict(np.load(path))
    assert np.allclose(head_predict(head, X[d.rows_va]), predict(model, d.rows_va), atol=1e-4), f"{name}: numpy head differs from torch head (val)"
    rng = np.random.default_rng(CFG["seed"])
    emb_rand = (mean + std * rng.standard_normal((256, cfg["dim"]))).astype(np.float32)
    with torch.no_grad():
        p_torch = torch.sigmoid(model(torch.from_numpy(((emb_rand - mean) / std).astype(np.float32)).to(DEVICE))).cpu().numpy()
    max_diff = float(np.abs(head_predict(head, emb_rand) - p_torch).max())
    assert max_diff < 1e-4, f"{name}: numpy head differs from torch head (random input): {max_diff}"
    return dict(val_auc_first5s=float(best["first"]), val_auc_clip_max=float(best["clip_max"]), best_epoch=int(best["best_epoch"]),
                val_classes=int(best["n_classes"]), soundscape_loss_weight_share=share, loss_weight_source=w_src, loss_weight_soundscape=w_sound,
                head_file=path.name, numpy_vs_torch_max_diff=max_diff, seconds=time.time() - t_start)

r = train_run(PERCH, d, d.E, T32, soft)
print(f"RESULT [perch/a1distill] first-5s {r['val_auc_first5s']:.4f} | clip-max {r['val_auc_clip_max']:.4f} | best ep {r['best_epoch']} | "
      f"head {r['seconds']:.0f}s | soundscape weight share {r['soundscape_loss_weight_share']:.4f}", flush=True)

## Final outputs

In [ ]:
json.dump(species, open(OUT / "species.json", "w"))
json.dump(dict(best_head="mlp", encoder="perch", teacher_kernel=TEACHER_KERNEL), open(OUT / "head_cfg.json", "w"), indent=1)
total = time.time() - T_NOTEBOOK
results = dict(config=dict(teacher=f"{TEACHER_KERNEL}/model.pth, unadapted, full eval, float32, no AMP", teacher_batch=TEACHER_BATCH,
                           teacher_power=TEACHER_POWER, teacher_floor=TEACHER_FLOOR, soundscape_loss_weight=SOUNDSCAPE_WEIGHT,
                           epochs=CFG["epochs"], seed=CFG["seed"], device=DEVICE, device_name=DEVICE_NAME, student_amp=False),
               teacher=teacher_stats, student=r, prep_seconds=d.prep_s, windows_seconds=WINDOWS_S, teacher_model_seconds=TEACHER_MODEL_S,
               teacher_seconds=TEACHER_S, total_seconds=total)
json.dump(to_py(results), open(OUT / "a1distill_results.json", "w"), indent=1)
print(f"TIMING | prep {d.prep_s:.0f}s | windows {WINDOWS_S:.0f}s | teacher model {TEACHER_MODEL_S:.0f}s | student {r['seconds']:.0f}s | notebook total {total / 60:.1f} min")